In [0]:
print(spark.table("city_lens.bronze.sru_raw").select("raw_xml").first()["raw_xml"][:5000])

In [0]:
# Cell 1: parse bronze XML into columns with a UDF
import xml.etree.ElementTree as ET
from pyspark.sql.functions import udf, col, explode_outer
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, ArrayType

CATALOG = "city_lens"
BRONZE_TABLE = f"{CATALOG}.bronze.sru_raw"

def local(tag):
    return tag.split("}")[-1]

def first_text(root, name):
    for e in root.iter():
        if local(e.tag) == name and e.text:
            return e.text.strip()
    return None

def all_text(root, name):
    return [e.text.strip() for e in root.iter() if local(e.tag) == name and e.text]

notice_schema = StructType([
    StructField("notice_id",      StringType()),
    StructField("title",          StringType()),
    StructField("rubriek",        StringType()),
    StructField("activities",     ArrayType(StringType())),
    StructField("abstract",       StringType()),
    StructField("creator",        StringType()),
    StructField("publication",    StringType()),
    StructField("modified",       StringType()),
    StructField("available",      StringType()),
    StructField("lat",            DoubleType()),
    StructField("lon",            DoubleType()),
    StructField("location_label", StringType()),
    StructField("municipality",   StringType()),
    StructField("url",            StringType()),
])

@udf(returnType=notice_schema)
def parse_notice(raw_xml):
    root = ET.fromstring(raw_xml)
    lat = lon = None
    point = first_text(root, "locatiepunt")          # "52.05 4.27"
    if point:
        parts = point.split()
        if len(parts) == 2:
            lat, lon = float(parts[0]), float(parts[1])
    return (
        first_text(root, "identifier"),
        first_text(root, "title"),
        first_text(root, "type"),
        all_text(root, "activiteit"),
        first_text(root, "abstract"),
        first_text(root, "creator"),
        first_text(root, "publicatienaam"),
        first_text(root, "modified"),
        first_text(root, "available"),
        lat, lon,
        first_text(root, "geometrielabel"),
        first_text(root, "ligtInGemeente"),
        first_text(root, "preferredUrl"),
    )

# Only the latest bronze load (bronze is idempotent per day)
latest = spark.sql(f"SELECT max(ingest_date) AS d FROM {BRONZE_TABLE}").first()["d"]

parsed = (
    spark.table(BRONZE_TABLE)
         .where(col("ingest_date") == latest)
         .withColumn("n", parse_notice(col("raw_xml")))
         .select("city_key", "ingest_ts", "n.*")
)

print("Latest bronze date:", latest, "| rows:", parsed.count())
display(parsed.limit(5))

In [0]:
# Cell 2: profile. Which types and activities exist?
display(
    parsed.withColumn("activity", explode_outer("activities"))
          .groupBy("rubriek", "activity")
          .count()
          .orderBy(col("count").desc())
)
display(parsed.where(col("lat").isNull()).groupBy("rubriek").count())

In [0]:
# Cell 3: clean types, categorise, keep the newest version per notice
from pyspark.sql import functions as F
from pyspark.sql.window import Window

title = F.lower(F.coalesce(col("title"), F.lit("")))
acts  = F.lower(F.concat_ws(",", col("activities")))
rub   = F.lower(F.coalesce(col("rubriek"), F.lit("")))


category = (
    F.when((rub == "evenementenvergunning") | title.rlike("evenement"), "event_permit")
     .when(rub.startswith("verkeersbesluit") | title.rlike("verkeersbesluit"), "traffic_measure")
     .when(title.rlike("^toestemming voor"), "street_works")
     .when(acts.rlike("bouwen|slopen|ruimtelijke|brandveilig|reclame"), "building_works")
     .when(acts.rlike("kappen"), "tree_felling")
     .otherwise("other")
)

stage = (
    F.when(category == "traffic_measure", "decided")
     .when(title.rlike("niet[- ]vergunningplichtig"), "not_required")
     .when(title.rlike("geweigerd|ingetrokken|buiten behandeling"), "refused")
     .when(title.rlike("verleend|toegestaan|^toestemming"), "granted")
     .when(title.rlike("aanvraag|aangevraagd|ontvangen"), "applied")
     .when(title.rlike("melding"), "notified")
     .otherwise("unknown")
)

latest_version = Window.partitionBy("notice_id").orderBy(col("modified").desc(), col("updated_ts").desc())

notices = (
    parsed
    .withColumn("modified",     F.to_date("modified"))
    .withColumn("available",    F.to_date("available"))
    .withColumn("category",     category)
    .withColumn("stage",        stage)
    .withColumn("has_location", col("lat").isNotNull())
    .withColumnRenamed("ingest_ts", "updated_ts")
    .where(col("notice_id").isNotNull())
    .withColumn("rn", F.row_number().over(latest_version))
    .where(col("rn") == 1)
    .drop("rn")
)

print("Notices after dedup:", notices.count())
display(notices.groupBy("category", "stage").count().orderBy("category", F.desc("count")))

In [0]:
# Cell 4: MERGE into silver.notices (create the table on first run)
SILVER_TABLE = f"{CATALOG}.silver.notices"

if not spark.catalog.tableExists(SILVER_TABLE):
    notices.limit(0).write.format("delta").saveAsTable(SILVER_TABLE)   # empty table with the right schema

notices.createOrReplaceTempView("notices_updates")

spark.sql(f"""
    MERGE INTO {SILVER_TABLE} AS t
    USING notices_updates AS s
      ON t.notice_id = s.notice_id
    WHEN MATCHED AND s.modified >= t.modified THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

display(spark.sql(f"""
    SELECT category,
           count(*)                                    AS notices,
           sum(CASE WHEN has_location THEN 1 ELSE 0 END) AS on_map,
           min(available)                              AS first_pub,
           max(available)                              AS last_pub
    FROM {SILVER_TABLE}
    GROUP BY category
    ORDER BY notices DESC
"""))

In [0]:
display(
    spark.table("city_lens.silver.notices")
         .where("stage = 'unknown' AND category IN ('other', 'building_works')")
         .select("category", "rubriek", "title")
         .limit(30)
)